# Sesión 4 · Extensiones

**Curso MCP · servidores remotos** — notebook 4 de 4

El core de MCP es deliberadamente pequeño. Todo lo demás vive en **extensiones**: piezas
opcionales que cliente y servidor negocian, y que permiten que el protocolo crezca sin
romper a nadie.

| Bloque | Minutos |
|---|:--:|
| Qué es una extensión y cómo se negocia | 20 |
| Extensiones de autorización | 20 |
| MCP Apps: interfaz en la conversación | 35 |
| Llevarlo a producción | 15 |

In [ ]:
!pip install --quiet "mcp==2.0.0"

MI_URL = "https://curso-mcp-XXXXX.europe-west1.run.app/mcp"  # ← EDITAR
from mcp import Client

## 1. Qué es una extensión

Una extensión es **un bloque de funcionalidad opcional identificado por un nombre con prefijo
de proveedor**, al estilo de los paquetes de Java:

- `io.modelcontextprotocol/ui` — oficial, interfaces en la conversación
- `io.modelcontextprotocol/tasks` — oficial, trabajos asíncronos
- `com.tudominio/loquesea` — la tuya, sin miedo a colisiones

### La regla que lo gobierna todo

**Una extensión es siempre opcional, y ambos lados tienen que funcionar sin ella.**

Ese es el contrato. Un servidor que *exige* una extensión para lo básico está roto para la
mitad del ecosistema. Lo que ofrece debe ser aditivo: mejor experiencia si el otro lado la
tiene, experiencia correcta si no.

### Cómo se negocia

Sin apretón de manos previo. Cada lado declara lo suyo:

**El cliente**, en cada petición, dentro de `_meta`:

```json
"io.modelcontextprotocol/clientCapabilities": {
  "extensions": { "io.modelcontextprotocol/ui": { "mimeTypes": ["text/html;profile=mcp-app"] } }
}
```

**El servidor**, en su respuesta a `server/discover`:

```json
"capabilities": { "tools": {}, "extensions": { "io.modelcontextprotocol/ui": {} } }
```

La intersección de ambas listas es lo que está en juego en esa conversación.

In [ ]:
import httpx, json

CABECERAS = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
    "MCP-Protocol-Version": "2026-07-28",
    "Mcp-Method": "server/discover",
    "Mcp-Name": "",   # el recurso al que apunta; vacío si no apunta a ninguno
}
META = {
    "io.modelcontextprotocol/protocolVersion": "2026-07-28",
    "io.modelcontextprotocol/clientCapabilities": {},
}

r = httpx.post(MI_URL, json={
    "jsonrpc": "2.0", "id": 1, "method": "server/discover", "params": {"_meta": META},
}, headers=CABECERAS, timeout=30).json()

print(json.dumps(r["result"]["capabilities"], indent=2, ensure_ascii=False))

## 2. Extensiones de autorización

La sesión anterior cubrió el OAuth del core: una persona delante, un navegador, un consentimiento.
Pero no todo el que consulta datos es una persona.

### `oauth-client-credentials`

Máquina a máquina: **sin navegador y sin usuario**. Un job nocturno que refresca un informe no
tiene a nadie a quien enseñarle una pantalla de consentimiento.

Encaja de forma natural con las cuentas de servicio de GCP, que es justamente cómo se
autentica un proceso automático en tu proyecto.

```python
from mcp.client.auth.extensions.client_credentials import ClientCredentialsProvider

proveedor = ClientCredentialsProvider(
    token_url="https://oauth2.googleapis.com/token",
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scopes=["bigquery.read"],
)
async with Client(MI_URL, auth=proveedor) as c:
    ...
```

### `enterprise-managed-authorization`

Para organizaciones donde el acceso lo decide un proveedor de identidad central y no cada
servidor por su cuenta. El servidor delega la política; TI la administra en un solo sitio.

> **Estado real, a agosto de 2026:** la matriz oficial de soporte muestra que estas dos
> extensiones apenas tienen clientes que las implementen, frente a MCP Apps que la soportan
> once. Que estén especificadas no significa que tu host las hable: **comprueba antes de
> construir sobre ellas**.

## 3. MCP Apps: interfaz dentro de la conversación

`io.modelcontextprotocol/ui` permite que un tool devuelva **una interfaz**, no solo texto.
Una tabla navegable, un gráfico, un selector.

Y la buena noticia para nosotros: **viene incluida en el SDK de Python**. Sin npm, sin build
de frontend. En TypeScript y C# es un paquete aparte; aquí no.

### Cómo se monta

Un widget es un HTML que el servidor sirve como recurso `ui://`, y un tool que lo referencia:

```python
from mcp.server.apps import Apps

apps = Apps()

@apps.tool(resource_uri="ui://curso-mcp/tabla.html", title="Consultar con tabla")
def consultar_con_tabla(sql: str, ctx: Context) -> dict:
    filas = bq.consultar(sql)
    return {"filas": filas, "total": len(filas)}

apps.add_html_resource("ui://curso-mcp/tabla.html", HTML_WIDGET)
mcp = MCPServer("curso-mcp-bigquery", extensions=[apps])
```

El host descarga ese HTML, lo mete en un **iframe aislado** y le pasa el resultado del tool.

In [ ]:
async def ver_widget():
    async with Client(MI_URL) as c:
        recursos = await c.list_resources()
        uis = [r for r in recursos.resources if str(r.uri).startswith("ui://")]
        print("Recursos de interfaz:", [str(r.uri) for r in uis])

        html = await c.read_resource("ui://curso-mcp/tabla.html")
        print("\nTamaño del widget:", len(html.contents[0].text), "bytes")

await ver_widget()

In [ ]:
# El mismo widget, renderizado aquí dentro con datos de mentira.
# Es lo que verá el usuario en Claude Desktop.
from IPython.display import HTML, display
import json as _json

async def previsualizar():
    async with Client(MI_URL) as c:
        r = await c.read_resource("ui://curso-mcp/tabla.html")
        html = r.contents[0].text

    datos = {"filas": [
        {"estacion": "Congress & 8th", "viajes": 1204},
        {"estacion": "Riverside & Lamar", "viajes": 987},
        {"estacion": "Guadalupe & 21st", "viajes": 764},
    ]}
    inyeccion = f"<script>window.postMessage({_json.dumps(datos)}, '*')</script>"
    display(HTML(html + inyeccion))

await previsualizar()

### Degradación elegante, en la práctica

Vuelve a mirar lo que devuelve el tool:

```python
return {"filas": filas, "total": len(filas)}
```

**No hay dos ramas de código.** No hay un `if el_host_soporta_ui`. Devuelve datos bien
formados, y con eso el widget pinta una tabla y un host sin interfaces enseña el JSON, que
sigue siendo perfectamente útil.

Cuando de verdad necesites ramificar —porque la versión rica y la pobre difieran de fondo—
existe `client_supports_apps(ctx)`. Pero si te encuentras usándolo mucho, probablemente el
payload esté mal diseñado.

## 4. Construir tu propia extensión

El SDK trae `mcp.server.extension.Extension`, y una extensión puede aportar exactamente cuatro
cosas: **tools**, **resources**, **métodos nuevos** y **un interceptor de `tools/call`**.

Ese conjunto cerrado es deliberado: una extensión amplía el protocolo, no lo reescribe.

En `curso_mcp/extension_auditoria.py` tienes una completa y pequeña, que registra cada llamada
y añade un método para consultarla:

```python
IDENTIFICADOR = "com.codecrypto.academy/auditoria"

class Auditoria(Extension):
    identifier = IDENTIFICADOR

    def settings(self) -> dict:
        return {"persistente": False, "limiteRegistro": 200}

    async def intercept_tool_call(self, params, ctx, call_next):
        self.anotaciones.append(Anotacion(params.name, dict(params.arguments or {})))
        return await call_next(ctx)     # interceptar no es secuestrar

    def methods(self):
        return (MethodBinding(method=f"{IDENTIFICADOR}.listar", ...),)
```

In [ ]:
# Pruébala en memoria, sin desplegar nada.
from mcp.server.mcpserver import MCPServer
from curso_mcp.extension_auditoria import Auditoria

async def probar_extension():
    auditoria = Auditoria()
    servidor = MCPServer("demo", extensions=[auditoria])

    @servidor.tool(description="Suma dos números.")
    def sumar(a: int, b: int) -> int:
        return a + b

    async with Client(servidor) as c:
        await c.call_tool("sumar", {"a": 2, "b": 3})
        await c.call_tool("sumar", {"a": 10, "b": 1})

    for anotacion in auditoria.anotaciones:
        print(anotacion.herramienta, anotacion.argumentos)

await probar_extension()

> **Ejercicio grande (material de ampliación).** Implementa la extensión oficial **Tasks**
> desde su especificación. No viene en el SDK 2.0.0, así que se escribe entera: `tasks/get`
> para consultar estado, `tasks/update` para mandar información a mitad de vuelo, y handles
> duraderos. Es el mejor ejercicio del curso porque te obliga a leer una spec y llevarla a
> código, que es exactamente lo que harás cuando salga la próxima revisión.

## 5. Llevarlo a producción

Cuatro cosas que muerden en cuanto hay usuarios de verdad:

**Observabilidad.** El SDK 2.0 trae OpenTelemetry integrado. Cloud Run exporta a Cloud Trace
sin que escribas nada; lo que hay que decidir es qué atributos añades a los spans.

**Límites de tamaño.** El transporte rechaza cuerpos de más de **4 MiB** con un `413`. Y por
arriba, los hosts truncan: alrededor de **150.000 caracteres** en claude.ai y Claude Desktop,
unos **25.000 tokens** en Claude Code. Una consulta que devuelve 10.000 filas no llega entera
a ningún sitio.

**Paginación en vez de volcado.** Devuelve las primeras filas, di cuántas hay en total y deja
que pidan más. Un `SELECT *` sin `LIMIT` es un fallo de diseño del tool, no del usuario.

**Coste.** Cada llamada es dinero en BigQuery. Cuota por identidad, `ttlMs` generoso en lo que
cambia poco, y el dry run del módulo 2 como cinturón.

In [ ]:
# Radiografía final de tu servidor.
async def resumen():
    async with Client(MI_URL) as c:
        tools = await c.list_tools()
        recursos = await c.list_resources()
        prompts = await c.list_prompts()
        print(f"Tools:     {len(tools.tools)}")
        print(f"Resources: {len(recursos.resources)}")
        print(f"Prompts:   {len(prompts.prompts)}")
        print(f"\nInstrucciones que lee el modelo:\n{c.instructions}")

await resumen()

## Hasta aquí el curso

Tienes un servidor MCP en Cloud Run que expone BigQuery con tools, resources y prompts, pide
confirmación antes de gastar, informa de su progreso, exige identidad y traduce esa identidad
a permisos reales, sirve una interfaz propia y carga una extensión que has escrito tú.

### Material de ampliación

Con el mismo servidor ya desplegado, por tu cuenta:

- **Implementar Tasks** desde la especificación.
- **`search` + `execute`** cuando la superficie pasa de quince acciones.
- **Paginación y presupuesto de payload** para consultas grandes.
- **Protección del servicio**: cuotas, abuso y control de gasto.

### Para seguir

- Especificación `2026-07-28`: https://modelcontextprotocol.io/specification/2026-07-28/
- Extensiones oficiales: https://modelcontextprotocol.io/extensions/overview
- Matriz de soporte por cliente: https://modelcontextprotocol.io/extensions/client-matrix

## Limpieza final: borrar lo que has creado

Un servicio de Cloud Run desplegado sigue existiendo hasta que lo borras, y el despliegue deja
más rastro del que parece:

| Artefacto | Qué es | ¿Cuesta dinero? |
|---|---|---|
| **Servicio de Cloud Run** | Tu servidor con su URL pública | Solo al recibir peticiones, pero **sigue accesible** |
| **Imágenes en Artifact Registry** | Cada despliegue sube una imagen nueva | Sí, por almacenamiento |
| **Bucket de staging de Cloud Build** | El código fuente que subiste | Sí, poco |

Lo que de verdad importa no es el coste, que es de céntimos: es que **mientras el servicio
exista con `--allow-unauthenticated`, cualquiera con la URL puede lanzarte consultas a
BigQuery**, y esas sí se facturan.

> ### ⚠️ El repositorio de imágenes es compartido
>
> `cloud-run-source-deploy` lo usan **todos** los servicios del proyecto desplegados con
> `--source`. Borrar el repositorio entero destruiría las imágenes de los demás. La celda de
> abajo borra **solo** la imagen de este curso. Nunca hagas
> `gcloud artifacts repositories delete cloud-run-source-deploy`.

Al terminar el curso, este es el momento de dejarlo todo como estaba.

In [ ]:
# Borra el servicio y la imagen de ESTE curso. No toca nada más del proyecto.
PROYECTO = "codecrypto-ai"   # ← EDITAR
REGION = "europe-west1"
SERVICIO = "curso-mcp"

# 1. El servicio: esto apaga la URL pública
!gcloud run services delete {SERVICIO} --project {PROYECTO} --region {REGION} --quiet

# 2. Solo la imagen de este servicio, no el repositorio que la contiene
!gcloud artifacts docker images delete \
  {REGION}-docker.pkg.dev/{PROYECTO}/cloud-run-source-deploy/{SERVICIO} \
  --project {PROYECTO} --delete-tags --quiet

In [ ]:
# Comprobación: el servicio ya no está y las imágenes de otros siguen ahí.
print("Servicios que quedan en la región:")
!gcloud run services list --project {PROYECTO} --region {REGION} --format="value(metadata.name)"

print("\nImágenes en el repositorio compartido (no debe salir curso-mcp):")
!gcloud artifacts docker images list \
  {REGION}-docker.pkg.dev/{PROYECTO}/cloud-run-source-deploy \
  --project {PROYECTO} --format="value(package)" 2>/dev/null | sed 's|.*/||' | sort -u